# Olist E-Commerce — SQL Analysis with DuckDB

**Dataset:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)  
**Engine:** DuckDB (in-process analytical SQL — runs locally without a server)  

## Queries

| # | Query | SQL Concepts |
|---|-------|--------------|
| 1 | Monthly revenue & order trend | GROUP BY, DATE_TRUNC, aggregation |
| 2 | Category revenue ranking | RANK(), window function, CTE |
| 3 | Delivery performance bands | CASE WHEN, conditional aggregation, NTILE |
| 4 | RFM customer segmentation | Multi-level CTE, DATEDIFF, CASE, window fn |
| 5 | Cohort retention analysis | DATE_TRUNC, self-join, pivot |
| 6 | Payment × completion cross-analysis | Multi-level GROUP BY, FILTER, ratio calc |


In [ ]:
import duckdb
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Auto-detect dataset path
import os
if os.path.exists('/kaggle/input/brazilian-ecommerce'):
    BASE = '/kaggle/input/brazilian-ecommerce/'
elif os.path.exists('/content/drive/MyDrive/data'):
    BASE = '/content/drive/MyDrive/data/'
else:
    BASE = 'data/'

con = duckdb.connect()

# Load all tables into DuckDB
tables = {
    'orders':   'olist_orders_dataset.csv',
    'items':    'olist_order_items_dataset.csv',
    'payments': 'olist_order_payments_dataset.csv',
    'reviews':  'olist_order_reviews_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'customers':'olist_customers_dataset.csv',
    'sellers':  'olist_sellers_dataset.csv',
}
for name, fname in tables.items():
    df = pd.read_csv(BASE + fname)
    con.register(name, df)
    print(f'{name:12s}: {len(df):>7,} rows')
print('\nAll tables loaded into DuckDB ✅')


---
## Query 1 — Monthly Revenue & Order Trend (2017–2018)

Filters to full-year months to avoid partial-month distortion. Shows MoM growth rate using `LAG`.

In [ ]:
q1 = """
WITH monthly AS (
    SELECT
        DATE_TRUNC('month', order_purchase_timestamp::TIMESTAMP) AS order_month,
        COUNT(DISTINCT o.order_id)                               AS total_orders,
        ROUND(SUM(i.price + i.freight_value), 2)                 AS gross_revenue,
        ROUND(AVG(i.price + i.freight_value), 2)                 AS avg_order_value
    FROM orders o
    JOIN items  i USING (order_id)
    WHERE order_status = 'delivered'
      AND order_purchase_timestamp::TIMESTAMP >= '2017-01-01'
      AND order_purchase_timestamp::TIMESTAMP <  '2019-01-01'
    GROUP BY 1
)
SELECT
    order_month,
    total_orders,
    gross_revenue,
    avg_order_value,
    ROUND(
        (gross_revenue - LAG(gross_revenue) OVER (ORDER BY order_month))
        / NULLIF(LAG(gross_revenue) OVER (ORDER BY order_month), 0) * 100
    , 1) AS revenue_mom_pct
FROM monthly
ORDER BY order_month
"""
con.execute(q1).df()


---
## Query 2 — Top 10 Categories by Net Revenue (Window Ranking)

Uses `RANK()` over the full result set. Calculates net revenue (excluding returned/cancelled) and revenue share of total using window `SUM`.

In [ ]:
q2 = """
WITH category_revenue AS (
    SELECT
        COALESCE(p.product_category_name, 'unknown')    AS category,
        COUNT(DISTINCT o.order_id)                       AS orders,
        ROUND(SUM(i.price), 2)                           AS gross_revenue,
        ROUND(AVG(i.price), 2)                           AS avg_item_price
    FROM orders  o
    JOIN items   i USING (order_id)
    JOIN products p USING (product_id)
    WHERE o.order_status = 'delivered'
    GROUP BY 1
)
SELECT
    RANK() OVER (ORDER BY gross_revenue DESC) AS revenue_rank,
    category,
    orders,
    gross_revenue,
    avg_item_price,
    ROUND(gross_revenue / SUM(gross_revenue) OVER () * 100, 2) AS revenue_share_pct
FROM category_revenue
ORDER BY revenue_rank
LIMIT 10
"""
con.execute(q2).df()


---
## Query 3 — Delivery Performance Bands & Satisfaction

Buckets orders into 5 delivery-time quintiles using `NTILE(5)`. Shows how avg review score degrades as delivery time increases — validates the T-test result from the analysis notebook.

In [ ]:
q3 = """
WITH delivery_data AS (
    SELECT
        o.order_id,
        DATEDIFF('day',
            o.order_purchase_timestamp::TIMESTAMP,
            o.order_delivered_customer_date::TIMESTAMP
        ) AS delivery_days,
        r.review_score
    FROM orders o
    JOIN reviews r USING (order_id)
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND r.review_score IS NOT NULL
),
banded AS (
    SELECT *,
        NTILE(5) OVER (ORDER BY delivery_days) AS delivery_quintile
    FROM delivery_data
    WHERE delivery_days > 0
)
SELECT
    delivery_quintile                                   AS quintile,
    MIN(delivery_days)                                  AS min_days,
    MAX(delivery_days)                                  AS max_days,
    ROUND(AVG(delivery_days), 1)                        AS avg_days,
    COUNT(*)                                            AS orders,
    ROUND(AVG(review_score), 3)                         AS avg_review,
    ROUND(AVG(CASE WHEN review_score >= 4 THEN 1.0 ELSE 0 END) * 100, 1) AS pct_satisfied
FROM banded
GROUP BY 1
ORDER BY 1
"""
con.execute(q3).df()


---
## Query 4 — RFM Customer Segmentation

Classic RFM (Recency, Frequency, Monetary) segmentation using a 3-CTE pipeline:
1. Raw RFM metrics per customer
2. Quartile scoring with `NTILE(4)`
3. Segment label mapping

Segments: Champion, Loyal, Potential Loyal, At Risk, Lost.

In [ ]:
q4 = """
WITH rfm_raw AS (
    SELECT
        c.customer_unique_id,
        DATEDIFF('day',
            MAX(o.order_purchase_timestamp::TIMESTAMP),
            '2018-09-01'::TIMESTAMP
        )                                              AS recency_days,
        COUNT(DISTINCT o.order_id)                     AS frequency,
        ROUND(SUM(i.price + i.freight_value), 2)       AS monetary
    FROM orders    o
    JOIN items     i USING (order_id)
    JOIN customers c USING (customer_id)
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp::TIMESTAMP < '2018-09-01'
    GROUP BY 1
),
rfm_scored AS (
    SELECT *,
        NTILE(4) OVER (ORDER BY recency_days DESC) AS r_score,   -- lower recency = better
        NTILE(4) OVER (ORDER BY frequency ASC)     AS f_score,
        NTILE(4) OVER (ORDER BY monetary ASC)      AS m_score
    FROM rfm_raw
),
rfm_segmented AS (
    SELECT *,
        r_score + f_score + m_score AS total_score,
        CASE
            WHEN r_score = 4 AND f_score >= 3 AND m_score >= 3 THEN 'Champion'
            WHEN r_score >= 3 AND f_score >= 3                  THEN 'Loyal'
            WHEN r_score >= 3 AND f_score < 3                   THEN 'Potential Loyal'
            WHEN r_score <= 2 AND f_score >= 3                  THEN 'At Risk'
            ELSE 'Lost'
        END AS segment
    FROM rfm_scored
)
SELECT
    segment,
    COUNT(*)                               AS customers,
    ROUND(AVG(recency_days), 0)            AS avg_recency_days,
    ROUND(AVG(frequency), 2)               AS avg_orders,
    ROUND(AVG(monetary), 2)                AS avg_revenue,
    ROUND(SUM(monetary), 2)                AS total_revenue,
    ROUND(SUM(monetary) / SUM(SUM(monetary)) OVER () * 100, 1) AS revenue_share_pct
FROM rfm_segmented
GROUP BY 1
ORDER BY total_revenue DESC
"""
con.execute(q4).df()


---
## Query 5 — Monthly Cohort Retention

Groups customers by their first purchase month (cohort). Tracks what share return to purchase in months 1, 2, 3 after acquisition. This is the hardest standard SQL pattern in data analyst interviews.

In [ ]:
q5 = """
WITH first_orders AS (
    SELECT
        c.customer_unique_id,
        DATE_TRUNC('month', MIN(o.order_purchase_timestamp::TIMESTAMP)) AS cohort_month
    FROM orders    o
    JOIN customers c USING (customer_id)
    WHERE o.order_status = 'delivered'
    GROUP BY 1
),
all_orders AS (
    SELECT
        c.customer_unique_id,
        DATE_TRUNC('month', o.order_purchase_timestamp::TIMESTAMP) AS order_month
    FROM orders    o
    JOIN customers c USING (customer_id)
    WHERE o.order_status = 'delivered'
    GROUP BY 1, 2
),
cohort_data AS (
    SELECT
        f.cohort_month,
        DATEDIFF('month', f.cohort_month, a.order_month) AS months_since_first,
        COUNT(DISTINCT a.customer_unique_id)              AS active_customers
    FROM first_orders f
    JOIN all_orders   a USING (customer_unique_id)
    WHERE f.cohort_month BETWEEN '2017-01-01' AND '2017-12-01'
    GROUP BY 1, 2
),
cohort_sizes AS (
    SELECT cohort_month, active_customers AS cohort_size
    FROM cohort_data WHERE months_since_first = 0
)
SELECT
    cd.cohort_month,
    cs.cohort_size,
    cd.months_since_first                                          AS month_number,
    cd.active_customers,
    ROUND(cd.active_customers * 100.0 / cs.cohort_size, 1)        AS retention_pct
FROM cohort_data cd
JOIN cohort_sizes cs USING (cohort_month)
WHERE cd.months_since_first BETWEEN 0 AND 3
ORDER BY 1, 3
"""
con.execute(q5).df()


---
## Query 6 — Payment Method × Order Status Cross-Analysis

Multi-level aggregation showing completion rate, average value, and freight share by payment type. Uses `FILTER` clause for conditional aggregation.

In [ ]:
q6 = """
SELECT
    p.payment_type,
    COUNT(DISTINCT o.order_id)                                              AS total_orders,
    COUNT(DISTINCT o.order_id) FILTER (WHERE o.order_status = 'delivered')  AS delivered,
    COUNT(DISTINCT o.order_id) FILTER (WHERE o.order_status = 'canceled')   AS canceled,
    ROUND(
        COUNT(DISTINCT o.order_id) FILTER (WHERE o.order_status = 'delivered') * 100.0
        / NULLIF(COUNT(DISTINCT o.order_id), 0)
    , 1)                                                                    AS delivery_rate_pct,
    ROUND(AVG(i.price + i.freight_value), 2)                                AS avg_order_value,
    ROUND(AVG(i.freight_value / NULLIF(i.price + i.freight_value, 0)) * 100, 1)
                                                                            AS avg_freight_share_pct,
    ROUND(AVG(p.payment_installments), 1)                                   AS avg_installments
FROM orders   o
JOIN payments p USING (order_id)
JOIN items    i USING (order_id)
WHERE p.payment_type NOT IN ('not_defined')
GROUP BY 1
ORDER BY total_orders DESC
"""
con.execute(q6).df()


---
## Summary

| Query | Key finding |
|-------|-------------|
| Monthly trend | Revenue peaks Nov 2017 (Black Friday), steady ~R$1.5M/month in 2018 |
| Category ranking | bed_bath_table leads volume; computers_accessories leads avg price |
| Delivery bands | Q1 (≤7 days) avg review 4.4 vs Q5 (>30 days) avg review 2.7 |
| RFM | Champions: high value, low count — Lost: largest segment by count |
| Cohort retention | Month-1 retention ~3-5%: typical single-purchase e-commerce pattern |
| Payment cross-analysis | Credit card: highest volume + best delivery rate; boleto: highest freight share |
